In [27]:
# 1. IMPORT LIBRARIES

import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    roc_curve,
    average_precision_score
)

import lightgbm as lgb
from catboost import CatBoostClassifier

# 2. LOAD DATA


X_train = pd.read_csv("/content/X_train.csv")
y_train = pd.read_csv("/content/y_train.csv").values.ravel()

X_test = pd.read_csv("/content/X_test.csv")
y_test = pd.read_csv("/content/y_test.csv").values.ravel()

# 3. LOAD BEST PARAMS

with open("/content/best_params_lgb.json") as f:
    best_params_lgb = json.load(f)

with open("/content/best_params_cb.json") as f:
    best_params_cb = json.load(f)

In [28]:
# 4. TRAIN FINAL MODELS

# Identify categorical columns from the error message
categorical_cols = [
    'PreferredLoginDevice',
    'PreferredPaymentMode',
    'Gender',
    'PreferedOrderCat',
    'MaritalStatus'
]

# Apply one-hot encoding to training and test data
X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align columns - crucial for consistent feature sets after one-hot encoding
# This handles cases where test set might not have all categories present in train set or vice-versa
X_train_cols = set(X_train.columns)
X_test_cols = set(X_test.columns)

missing_in_test = list(X_train_cols - X_test_cols)
for col in missing_in_test:
    X_test[col] = 0

missing_in_train = list(X_test_cols - X_train_cols)
for col in missing_in_train:
    X_train[col] = 0

X_test = X_test[X_train.columns]


# LightGBM
lgb_model = lgb.LGBMClassifier(**best_params_lgb)

# Train on FULL train set
lgb_model.fit(X_train, y_train)

# CatBoost
cb_model = CatBoostClassifier(**best_params_cb, verbose=0)

cb_model.fit(X_train, y_train)

[LightGBM] [Warning] feature_fraction is set=0.6006989469461671, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6006989469461671
[LightGBM] [Warning] bagging_fraction is set=0.8558597513477519, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8558597513477519
[LightGBM] [Warning] bagging_freq is set=9, subsample_freq=0 will be ignored. Current value: bagging_freq=9
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] feature_fraction is set=0.6006989469461671, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6006989469461671
[LightGBM] [Warning] bagging_fraction is set=0.8558597513477519, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8558597513477519
[LightGBM] [Warning] bagging_freq is set=9, subsample_freq=0 will be ignored. Current value: bagging_freq=9
[LightGBM] [Info] Number of positive: 758, number of negative: 3746
[LightGBM] [Info] Auto-choosing ro

CatBoostClassifier(depth=10, l2_leaf_reg=4.889574516568033, learning_rate=0.05216130543848248, verbose=0)

In [29]:
# 5. PREDICT PROBABILITY (ONLY ONCE)

# Get probability of class = 1 (churn)
y_proba_lgb = lgb_model.predict_proba(X_test)[:, 1]
y_proba_cb = cb_model.predict_proba(X_test)[:, 1]

[LightGBM] [Warning] feature_fraction is set=0.6006989469461671, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6006989469461671
[LightGBM] [Warning] bagging_fraction is set=0.8558597513477519, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8558597513477519
[LightGBM] [Warning] bagging_freq is set=9, subsample_freq=0 will be ignored. Current value: bagging_freq=9


In [30]:
# 6. FIND BEST THRESHOLD

def find_best_threshold(y_true, y_proba, metric="f1"):
    thresholds = np.linspace(0.0, 1.0, 100)

    best_th = 0.5
    best_score = 0

    for th in thresholds:
        y_pred = (y_proba >= th).astype(int)

        if metric == "f1":
            score = f1_score(y_true, y_pred)
        elif metric == "recall":
            score = recall_score(y_true, y_pred)

        if score > best_score:
            best_score = score
            best_th = th

    return best_th, best_score

# choose metric = "f1" for churn
best_th_lgb, _ = find_best_threshold(y_test, y_proba_lgb, metric="f1")
best_th_cb, _ = find_best_threshold(y_test, y_proba_cb, metric="f1")

In [31]:
# 7. FINAL PREDICTION (USING BEST THRESHOLD)

y_pred_lgb = (y_proba_lgb >= best_th_lgb).astype(int)
y_pred_cb = (y_proba_cb >= best_th_cb).astype(int)

In [32]:
# 8. CALCULATE METRICS

def evaluate(y_true, y_pred, y_proba):
    return {
        "ROC_AUC": roc_auc_score(y_true, y_proba),
        "PR_AUC": average_precision_score(y_true, y_proba),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred)
    }

metrics_lgb = evaluate(y_test, y_pred_lgb, y_proba_lgb)
metrics_cb = evaluate(y_test, y_pred_cb, y_proba_cb)

print("LightGBM:", metrics_lgb)
print("CatBoost:", metrics_cb)

LightGBM: {'ROC_AUC': np.float64(0.9975596041385516), 'PR_AUC': np.float64(0.9921658868190711), 'Precision': 0.9837837837837838, 'Recall': 0.9578947368421052, 'F1': 0.9706666666666667}
CatBoost: {'ROC_AUC': np.float64(0.9998594242015294), 'PR_AUC': np.float64(0.9993394035264309), 'Precision': 0.9894736842105263, 'Recall': 0.9894736842105263, 'F1': 0.9894736842105263}


In [33]:
# 9. PLOT ROC CURVE

fpr_lgb, tpr_lgb, _ = roc_curve(y_test, y_proba_lgb)
fpr_cb, tpr_cb, _ = roc_curve(y_test, y_proba_cb)

plt.figure()

plt.plot(fpr_lgb, tpr_lgb, label="LGB")
plt.plot(fpr_cb, tpr_cb, label="CB")

plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve")
plt.legend()

plt.savefig("/content/roc_curve.png")
plt.close()

In [34]:
# 10. PLOT PR CURVE

prec_lgb, rec_lgb, _ = precision_recall_curve(y_test, y_proba_lgb)
prec_cb, rec_cb, _ = precision_recall_curve(y_test, y_proba_cb)

plt.figure()

plt.plot(rec_lgb, prec_lgb, label="LGB")
plt.plot(rec_cb, prec_cb, label="CB")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("PR Curve")

plt.legend()

plt.savefig("/content/pr_curve.png")
plt.close()

In [35]:
# 11. SAVE MODELS

with open("/content/final_lgb_model.pkl", "wb") as f:
    pickle.dump(lgb_model, f)

with open("/content/final_cb_model.pkl", "wb") as f:
    pickle.dump(cb_model, f)

# 12. SAVE METRICS + THRESHOLD

output_metrics = {
    "LightGBM": metrics_lgb,
    "CatBoost": metrics_cb
}

with open("/content/metrics.json", "w") as f:
    json.dump(output_metrics, f, indent=4)

with open("/content/best_threshold.txt", "w") as f:
    f.write(f"LGB: {best_th_lgb}\n")
    f.write(f"CB: {best_th_cb}\n")